In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.style.use('seaborn-v0_8-darkgrid')

import yfinance as yf

spy = yf.download("SPY", start="2005-01-01", end="2024-12-31", auto_adjust=True)
vix = yf.download("^VIX", start="2005-01-01", end="2024-12-31", auto_adjust=True)

spy.columns = spy.columns.get_level_values(0)
vix.columns = vix.columns.get_level_values(0)

spy_close = spy["Close"]
vix_close = vix["Close"]

log_returns = np.log(spy_close / spy_close.shift(1)).dropna()
realized_vol = log_returns.rolling(window=21).std() * np.sqrt(252)

data = pd.DataFrame({
    "log_returns": log_returns,
    "realized_vol": realized_vol,
    "vix": vix_close
}).dropna()

returns_pct = data["log_returns"] * 100

print(f"Data loaded: {data.shape[0]} observations")
print(f"Date range: {data.index[0].date()} to {data.index[-1].date()}")

## Train/Test Split

In [ ]:
# 80/20 train/test split
split_idx = int(len(data) * 0.80)
split_date = data.index[split_idx]

train = data.iloc[:split_idx].copy()
test = data.iloc[split_idx:].copy()

train_returns_pct = train["log_returns"] * 100
test_returns_pct = test["log_returns"] * 100

print(f"Train: {train.index[0].date()} to {train.index[-1].date()} ({len(train)} days)")
print(f"Test:  {test.index[0].date()} to {test.index[-1].date()} ({len(test)} days)")
print(f"Split date: {split_date.date()}")